In [1]:
print("Starting imports...")
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float64, complex128
from numba.cuda import jit as cuda_jit
import math

print("Importing few...")
import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import FastKerrEccentricEquatorialFlux

from few.utils.geodesic import get_fundamental_frequencies

import os
import sys

# Change to the desired directory
os.chdir('/home/svu/e1498138/emri_search/work')

# Add it to Python path
sys.path.insert(0, '/home/svu/e1498138/emri_search/work')

print("Importing GWfuncs...")
import GWfuncs
# import gc
# import pickle
print("Importing cupy...")
import cupy as cp

print("Configuring few...")
# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

# print("Importing dynesty...")
# import dynesty

Starting imports...
Importing few...
Importing GWfuncs...
Importing cupy...
Configuring few...


In [2]:
from lisatools.sensitivity import *


In [3]:
few.has_backend('cuda12x')

True

In [4]:
from few.utils.constants import YRSID_SI, Gpc, MRSUN_SI


In [5]:
N_traj = 1e4 # change amount of points here 
T = 1 #yr
delta_T = T*YRSID_SI/N_traj 

In [6]:
use_gpu = True
force_backend = "cuda12x"  
print("Setting up waveform generator...")
# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": False, 
        "use_gpu" : use_gpu,
        "force_backend":force_backend
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": force_backend,
    # "use_gpu" : use_gpu
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": force_backend,
    # "assume_positive_m": True  # if we assume positive m, it will generate negative m for all m>0
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs = {
    "force_backend":force_backend,
    "pad_output": True,
    "separate_modes": True
    # "use_gpu" : use_gpu
}

print("Creating FastKerrEccentricEquatorialFlux...")
# Kerr eccentric flux
waveform_gen = FastKerrEccentricEquatorialFlux(
    inspiral_kwargs=inspiral_kwargs,
    amplitude_kwargs=amplitude_kwargs,
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs,
    use_gpu=use_gpu,
)

Setting up waveform generator...
Creating FastKerrEccentricEquatorialFlux...


In [7]:
# Parameters
m1 = 1e6 #M
m2 = 1e1 #mu
a = 0.5
p0 = 9.5
e0 = 0.2
theta = np.pi / 3.0 
phi = np.pi / 4.0  
dt = 10.0
xI0 = 1.0 
dist= 1

In [8]:
%%time
h = waveform_gen(
    m1, 
    m2,
    a, 
    p0, 
    e0, 
    xI0, 
    theta, 
    phi, 
    dist=dist, 
    dt=dt, 
    T=1
)


CPU times: user 24.6 s, sys: 420 ms, total: 25 s
Wall time: 27.2 s


In [ ]:
%%time
h = waveform_gen(
    m1, 
    m2,
    a, 
    p0, 
    e0, 
    xI0, 
    theta, 
    phi, 
    dist=dist, 
    dt=dt, 
    T=1
)


CPU times: user 22.9 ms, sys: 2.14 ms, total: 25.1 ms
Wall time: 24.4 ms


In [10]:
%%time
h = waveform_gen(
    m1+2, 
    m2,
    a, 
    p0, 
    e0, 
    xI0, 
    theta, 
    phi, 
    dist=dist, 
    dt=dt, 
    T=1
)


CPU times: user 20.4 ms, sys: 3.91 ms, total: 24.3 ms
Wall time: 23.6 ms


In [11]:
%%time
h = waveform_gen(
    m1+2, 
    m2+0.2,
    a+0.02, 
    p0-0.3, 
    e0, 
    xI0, 
    theta, 
    phi, 
    dist=dist, 
    dt=dt, 
    T=1
)


CPU times: user 22.1 ms, sys: 2.01 ms, total: 24.1 ms
Wall time: 23.3 ms
